# S4 J5 — Workflow Engine

Ce notebook est généré à partir du Markdown source du jour.

## Objectifs

- Comprendre le rôle d'un workflow engine.
- Modéliser un graphe d'étapes.
- Exécuter un workflow déterministe.
- Observer retries, conditions, blocages et traces.

## Modèle mental

Un workflow engine transforme un processus métier en contrat exécutable : étapes, dépendances, statuts, approbations et traces.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT.name != "ai-engineering-bootcamp" and ROOT.parent != ROOT:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mini_framework.workflow import WorkflowEngine
from book.week04.day05.labs.workflow_engine_lab import build_support_workflow, register_support_handlers

engine = WorkflowEngine()
register_support_handlers(engine)
workflow = build_support_workflow()
engine.manifest(workflow)

## Exécution avec approbation

L'étape sensible `refund` peut s'exécuter lorsque l'approbation humaine est fournie.

In [ ]:
run = engine.run(
    workflow,
    input_payload={
        "message": "Urgent refund request for a duplicated charge",
        "customer_tier": "premium",
        "amount": 42.0,
    },
    approvals={"refund"},
    run_id="notebook_run_approved",
)
run.status, run.step_results["refund"].status, run.state.data["final_answer"]

## Exécution sans approbation

Le moteur bloque l'étape sensible et évite de produire une fausse finalisation.

In [ ]:
blocked_run = engine.run(
    workflow,
    input_payload={"message": "refund please", "amount": 42.0},
    approvals=set(),
    run_id="notebook_run_blocked",
)
blocked_run.status, blocked_run.step_results["refund"].status, blocked_run.step_results["final_answer"].status

## Lire la trace

In [ ]:
[(event.step, event.event, event.payload) for event in blocked_run.trace]

## Exercices

1. Ajoutez une étape sensible `notify_customer` après `final_answer`.
2. Ajoutez un test qui vérifie le blocage sans approbation.
3. Modifiez le workflow pour que `final_answer` puisse produire un résumé même si `refund` est bloqué.
4. Expliquez le risque métier associé à cette modification.